# 01 - Data preparation and pronunciation labels

All the offline work happens here. Afterwards training reads only memmaps, so
the GPU never waits on data.

| Stage | What it does | Time on a 4090 |
|---|---|---|
| A0 | manifest: scan, normalize, filter | 1 min |
| A0b | **diacritize with CATT-ECA** (own conda env) | about 20 min |
| A1 | CTC forced alignment to word spans | about 35 min |
| A3 | **derive readings** -> the decision point | about 2 min |
| A4 | MARBERTv2 teacher cache and head | about 8 min |
| A5 | Mimi encode to RVQ codes | about 30 min |

Run the cells in order, top to bottom.

## What changed, and why it matters

The first full run cost $15 and learned nothing, because labels came from
unsupervised clustering of acoustic word spans. That returned the channel's
subscribe pitch as "homographs" (الجرس, لايك, الوصف) while علم, مصر and دول each
got a single code. A mean-pooled speech embedding encodes speaking rate and
recording session far more strongly than vowel identity, so no threshold on that
signal can separate the two.

Labels now come from a **diacritizer**, which observes the vowels directly.
Stage A2 (span embeddings) is no longer on the critical path and is skipped.

Stage A3 is the decision point: it costs two minutes and tells you whether the
labels are sound *before* you spend money on training.

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"

# ---------------------------------------------------------------------------
# PICK YOUR EXPERIMENT HERE. This is the only line to change.
#
#   configs/exp0_small.yaml     2013 clips, ~2 GB   -> proves the pipeline,
#                                                     runs on a 6 GB GPU
#   configs/exp1_egyptian.yaml  15.6k clips, 68 h   -> the real run
# ---------------------------------------------------------------------------
CONFIG = "configs/exp0_small.yaml"

from adaptts.utils.config import load_config
from adaptts.utils.logging_utils import setup_logging
setup_logging()
cfg = load_config(CONFIG)
print("repo   :", REPO)
print("config :", CONFIG, "->", cfg.name)
print("dataset:", cfg.paths.hf_dataset_id)

# The homographs named in the brief, read from the probe file so the notebooks
# never hardcode a word list of their own.
import json as _json
PROBE_WORDS = sorted({
    w for _s in _json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]
    for w in _s["focus"].split(" / ") if w and w != "none"
})
print("probe  :", " ".join(PROBE_WORDS))

## Download the dataset

About 12 GB. To use a different corpus, change `paths.hf_dataset_id` and
`paths.dataset_dir` in the config.

In [ ]:
from huggingface_hub import snapshot_download

target = cfg.paths.dataset_dir
print("downloading to:", target)
snapshot_download(
    repo_id=cfg.paths.hf_dataset_id, repo_type="dataset",
    local_dir=target, max_workers=8,
)
print("done")

In [ ]:
import glob, os

wavs = glob.glob(os.path.join(target, "clips", "**", "*.wav"), recursive=True)
parquet = glob.glob(os.path.join(target, "**", "*.parquet"), recursive=True)
meta = os.path.join(target, "metadata")
print("wav clips     :", len(wavs))
print("parquet shards:", len(parquet))
print("metadata dir  :", os.listdir(meta) if os.path.isdir(meta) else "none")
if parquet and not wavs:
    print()
    print("This is a parquet dataset. The manifest stage below unpacks the")
    print("embedded audio to wav once, so later stages never decode it again.")

## Stage A0 - manifest

Normalizes every transcript once, so alignment, the teacher and training all see
byte-identical strings.

In [ ]:
!python scripts/preprocess.py --config $CONFIG --stage manifest

In [ ]:
import json, collections

rows = [json.loads(l) for l in open(cfg.paths.manifest_path, encoding="utf-8")]
hours = sum(r["duration"] for r in rows) / 3600
print(f"{len(rows)} utterances, {hours:.1f} hours")
print("splits:", collections.Counter(r["split"] for r in rows))
for r in rows[:3]:
    print(f'  [{r["duration"]:.1f}s] {r["text"][:70]}')

### Check the text normalization

Every transcript passed through the Egyptian normalizer. Numbers, dates and
Latin tokens should all be spoken words by now, with no digits left.

In [ ]:
import json, random

rows = [json.loads(l) for l in open(cfg.paths.manifest_path, encoding="utf-8")]
random.seed(0)
for r in random.sample(rows, min(8, len(rows))):
    print(f"[{r['duration']:5.1f}s] {r['text'][:100]}")

leftover = [r for r in rows if any(c.isdigit() for c in r["text"])]
print()
print(f"utterances still containing digits: {len(leftover)} of {len(rows)}")
for r in leftover[:3]:
    print("   ", r["text"][:100])

## Stage A0b - diacritize with CATT-ECA

This is the label source. It runs **in the CATT environment**, not this one, so
it goes through that interpreter explicitly rather than through `!python`.

The diacritics are a labelling device only. They tell us which reading each word
occurrence takes; the shipped model never sees a diacritic and you never type
one at inference. This is the same role the forced aligner plays for word spans.

About 20 minutes for 15.6k sentences. Runs once, then caches.

In [ ]:
import os, subprocess, sys, yaml

_raw = yaml.safe_load(open(CONFIG, encoding="utf-8"))
_base = yaml.safe_load(open("configs/base.yaml", encoding="utf-8"))
CATT_ROOT = (os.environ.get("CATT_ROOT")
             or _raw.get("paths", {}).get("catt_root")
             or _base["paths"].get("catt_root", ""))
CATT_PY = os.environ.get(
    "CATT_PY", os.path.expanduser("~/miniconda3/envs/CATT/bin/python")
)
print("catt_root  :", CATT_ROOT or "(not set)")
print("catt python:", CATT_PY)

assert CATT_ROOT and os.path.isdir(os.path.join(CATT_ROOT, "catt_tashkeel")), (
    "catt_tashkeel/ not found under catt_root. Upload the folder and set "
    "paths.catt_root in your config."
)
assert os.path.isfile(CATT_PY), (
    "CATT interpreter not found. Create the CATT env (notebook 00) or set CATT_PY."
)

In [ ]:
# Streams output so you can watch progress rather than waiting on a block.
import subprocess, sys, os

cmd = [CATT_PY, "scripts/diacritize.py", "--config", CONFIG,
       "--catt-root", CATT_ROOT, "--batch-size", "32"]
print(" ".join(cmd))
print()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1,
                     env=dict(os.environ, PYTHONIOENCODING="utf-8"))
for line in p.stdout:
    print(line.rstrip(), flush=True)
p.wait()
assert p.returncode == 0, f"diacritize failed with code {p.returncode}"

In [ ]:
# Inspect the diacritized output. The homographs should differ in their marks.
import json

rows = [json.loads(l) for l in open(cfg.paths.diacritized_path, encoding="utf-8")]
print(f"{len(rows)} diacritized utterances")
print()
for r in rows[:3]:
    print("plain:", r["text"][:80])
    print("diac :", r["diacritized"][:80])
    print()

# Token counts must match, or an occurrence would be mislabelled. The reading
# stage skips any mismatch rather than guessing, but a high rate means trouble.
bad = sum(1 for r in rows if len(r["text"].split()) != len(r["diacritized"].split()))
print(f"token-count mismatches: {bad} of {len(rows)} ({100*bad/max(len(rows),1):.1f}%)")

## Stage A1 - CTC forced alignment

Finds the time span of every word with no pronunciation lexicon. The Viterbi
alignment is implemented in-repo, so there is no Montreal Forced Aligner
dependency.

In [ ]:
!python scripts/preprocess.py --config $CONFIG --stage align

## Stage A3 - derive the readings

### This is the decision point of the whole project

Two minutes here decides whether training is worth paying for. The stage groups
every occurrence of every word by its vowel pattern and keeps a split only when
it survives the artifact filters and the context-agreement gate.

Span embeddings (the old stage A2) are not needed for labels and are skipped.

In [ ]:
!python scripts/preprocess.py --config $CONFIG --stage discover

In [ ]:
from adaptts.text.diacritics import ReadingLexicon

lex = ReadingLexicon.load(cfg.paths.reading_lexicon_path)
amb = lex.ambiguous_words
print(f"{len(lex)} word types, {len(amb)} with more than one reading")
print()

entries = sorted((lex.entries[w] for w in amb), key=lambda e: -e.total)
print(f"{'word':<16}{'readings':>9}{'uses':>7}  counts  examples")
print("-" * 74)
for e in entries[:40]:
    print(f"{e.word:<16}{e.n_codes:>9}{e.total:>7}  {e.counts}  {' '.join(e.examples)}")

In [ ]:
# The homographs named in the brief. These are the reason the project exists.
for w in PROBE_WORDS:
    e = lex.entries.get(w)
    k = lex.n_codes(w)
    ex = " ".join(e.examples) if e else ""
    print(f"{w:<10} -> {k} reading(s)  {'AMBIGUOUS' if k > 1 else 'single':<10} {ex}")

In [ ]:
# The failure mode from the first run: promo words must NOT be ambiguous.
# Clustering split these on speaking register. If any shows up here, the
# labels have regressed and training would waste money again.
PROMO = ["الجرس", "التعليقات", "لايك", "الوصف", "الرابط", "البلاي", "اكتبوه", "عندكم"]
bad = [w for w in PROMO if lex.n_codes(w) > 1]
for w in PROMO:
    k = lex.n_codes(w)
    print(f"{w:<14} {k} reading(s)" + ("   <-- REGRESSION" if k > 1 else ""))
print()
print("promo words clean" if not bad else f"PROBLEM: {bad} split again")

In [ ]:
# Read the sentences behind each reading. This is how you confirm the labels
# track meaning rather than an artifact of the diacritizer.
import json, collections

rows = [json.loads(l) for l in open(cfg.paths.diacritized_path, encoding="utf-8")]

WORD = PROBE_WORDS[0]      # change to inspect any ambiguous word
e = lex.entries.get(WORD)
if e is None or e.n_codes < 2:
    print(f"{WORD} has a single reading here; pick another from the table above.")
else:
    groups = collections.defaultdict(list)
    for r in rows:
        plain, diac = r["text"].split(), r["diacritized"].split()
        if len(plain) != len(diac):
            continue
        for w, d in zip(plain, diac):
            if w == WORD:
                c = e.code_of(d)
                if c >= 0:
                    groups[c].append((d, r["text"]))
    for c in sorted(groups):
        print()
        print(f"=== {WORD}  reading {c}  ({e.examples[c]})  "
              f"{len(groups[c])} occurrences ===")
        for d, t in groups[c][:6]:
            print(f"   {d:<14} {t[:80]}")

**How to read this.** Sentences under one reading should share a meaning: علم as
flag in one group, as science in the other. If the groups look mixed, raise
`discovery.min_pattern_count` in the config and rerun this stage with `--force`.

If the promo-word check above shows a regression, stop. Do not train.

## Stage A4 - teacher cache

One frozen MARBERTv2 pass over the corpus, cached as fp16. The teacher never
runs again, which is the main reason training is cheap.

In [ ]:
!python scripts/preprocess.py --config $CONFIG --stage teacher

## Stage A5 - Mimi codec encoding

Every clip becomes 8 RVQ streams at 12.5 Hz. A 10 second clip is 125 frames,
which is why generation is fast on a CPU.

In [ ]:
!python scripts/preprocess.py --config $CONFIG --stage codec

## Verify the cache is complete

In [ ]:
from adaptts.data.dataset import AdapTTSDataset, collate
from adaptts.text.vocab import CharVocab

vocab = CharVocab.load(cfg.paths.charvocab_path)
ds = AdapTTSDataset(cfg, "train", vocab, lex, need_codes=True, need_teacher=True)
b = collate(
    [ds[i] for i in range(4)], vocab.pad_id, cfg.discovery.max_codes_per_word,
    cfg.audio.n_quantizers, cfg.teacher.hidden_size,
)
for k, v in b.items():
    print(f"  {k:<20} {tuple(v.shape)}  {v.dtype}")
print()
print("ambiguous words in this batch:", int((b["n_codes"] > 1).sum()))
ds.close()

## The majority baseline

The number notebook 02 has to beat. If every ambiguous word were always given
its most common reading, this is the accuracy you would get for free. A context
encoder that scores at or below this has learned nothing, whatever the loss
curve looks like.

Write it down before training.

In [ ]:
best = sum(max(lex.entries[w].counts) for w in amb)
total = sum(lex.entries[w].total for w in amb)
baseline = best / max(total, 1)
print(f"ambiguous word types : {len(amb)}")
print(f"labelled occurrences : {total}")
print(f"MAJORITY BASELINE    : {baseline:.3f}")
print()
print("Notebook 02 must beat this by a clear margin, not by 1%.")
print()
print("Data is ready. Continue to 02_train_context.ipynb")